# Sales Prediction Web App

Building a complete **sales prediction web application** using a pretrained Random Forest model. You'll create a Python project with a virtual environment, load model components, set up a Streamlit interface for user inputs, preprocess those inputs, and generate sales predictions.

The steps mirror the tutorial on the platform:

- **Part I:** Create your project environment and install dependencies.
- **Part II:** Import the required libraries in a new `app.py` file.
- **Part III:** Load the trained model and preprocessing components from `rf_model.pkl`.
- **Part IV:** Build an interactive Streamlit interface with a sidebar and input fields.
- **Part V:** Add a "Predict" button that preprocesses inputs, runs the model, and displays results.
- **Part VI:** Launch the app with `streamlit run app.py`.

> **Note:** The transformers and gradio packages may not be available in this sandbox environment, so this notebook demonstrates the code structure. You should run it in Google Colab or locally with the required packages installed.


## Part I – Create Your Project Environment

1. **Open your terminal.**
2. **Create a new directory for your project** (e.g. `sales_prediction_app`).
3. **Navigate** into the new directory.
4. **Create and activate a virtual environment**. Use the appropriate commands for your operating system.

- On Windows:
  ```bash
  python -m venv venv
  venv\Scriptsctivate
  python -m pip install --upgrade pip
  python -m pip install -r requirements.txt
  ```
- On macOS/Linux:
  ```bash
  python3 -m venv venv
  source venv/bin/activate
  python -m pip install --upgrade pip
  python -m pip install -r requirements.txt
  ```

The `requirements.txt` file should list the necessary packages, such as `streamlit`, `pandas`, `numpy`, `scikit-learn`, and any other dependencies used in this exercise.


## Part II – Install and Import Required Libraries

After activating your environment, create a Python script (e.g. `app.py`) and import the required libraries:

```python
import streamlit as st
import pandas as pd
import numpy as np
import pickle
from sklearn.preprocessing import StandardScaler
```

These imports bring in Streamlit for the web UI, pandas and numpy for data handling, pickle for loading the model, and `StandardScaler` for scaling numeric features.


## Part III – Load the Trained ML Model

Load your pretrained model and preprocessing objects from `rf_model.pkl`. The file should contain a dictionary with the components you used during training (e.g. imputers, encoders, the Random Forest model, and any scalers):

```python
# Path to your saved model file
model_path = 'rf_model.pkl'

with open(model_path, 'rb') as f:
    components = pickle.load(f)

# Extract the components
num_imputer = components['num_imputer']   # Imputer for numerical columns
cat_imputer = components['cat_imputer']   # Imputer for categorical columns
encoder     = components['encoder']       # OneHotEncoder or similar
model       = components['model']         # Trained Random Forest model
scaler      = components.get('scaler', StandardScaler())  # Optional scaler

# List of categorical columns used during training
categorical_columns = components['categorical_columns']
```

Make sure `rf_model.pkl` contains all the necessary components. If you used a pipeline object instead of separate components, adjust the extraction accordingly.


## Part IV – Build the Streamlit App Interface

A clean and interactive UI helps users understand what they need to input. We'll display a header, a caption, and a sidebar with descriptions for each input field. We'll also arrange input widgets in three columns.

```python
# Title and caption
st.title("Sales Prediction Web App")
st.caption("Enter the store and product details to predict daily sales.")

# Sidebar with descriptions
description_text = (
    "- **Store Number**: Unique ID of the store
"
    "- **Product Family**: Category such as AUTOMOTIVE, BEAUTY, HOME AND KITCHEN I, etc.
"
    "- **Number of Items on Promotion**: Number of items on promotion within the store
"
    "- **State**: Province or region where the store is located
"
    "- **Transactions**: Number of transactions recorded
"
    "- **Store Type**: Type of store (e.g. D1, D2, D3)
"
    "- **Cluster**: Numeric cluster ID grouping similar stores
"
    "- **Month/Day/Day of Week**: Date components for the prediction
"
)
st.sidebar.header("Input Field Descriptions")
st.sidebar.markdown(description_text)

# Define a 3‑column layout for inputs
a, b, c = st.columns(3)

# Column A inputs
store_nbr = a.slider("Store Number", min_value=1, max_value=54, value=1)
family = a.selectbox("Product Family", ['AUTOMOTIVE', 'BEAUTY', 'HOME AND KITCHEN I', 'STATIONERY', 'GROCERY', 'CLEANING', 'FOODS'])
onpromotion = a.number_input("Number of Items on Promotion", min_value=0, value=0)

# Column B inputs
state = b.selectbox("State", [
    'Pichincha', 'Santo Domingo de los Tsachilas', 'Bolivar', 'Pastaza', 'Tungurahua',
    'Guayas', 'El Oro', 'Esmeraldas', 'Manabi'
])
transactions = b.number_input("Transactions", min_value=0, value=0)
store_type = b.selectbox("Store Type", ['D1', 'D2', 'D3'])
cluster = b.number_input("Cluster", min_value=0, value=0)

# Column C inputs
month = c.slider("Month", 1, 12, 1)
day = c.slider("Day", 1, 31, 1)
dayofweek = c.slider("Day of Week", 0, 6, 0)

# Collect inputs into a dictionary
input_data = {
    'store_nbr': store_nbr,
    'family': family,
    'onpromotion': onpromotion,
    'state': state,
    'transactions': transactions,
    'store_type': store_type,
    'cluster': cluster,
    'month': month,
    'day': day,
    'dayofweek': dayofweek,
}
```

This code sets up the layout and collects user inputs. Adjust the options and defaults to match your dataset.


## Part V – Create a Button to Trigger Predictions

When the user clicks the **Predict** button, we need to perform the same preprocessing steps used during training: impute missing values, encode categorical variables, scale numerical variables, and pass the processed data to the model. Here's how you can wire it up:

```python
if st.button("Predict"):
    # Convert inputs to DataFrame
    input_df = pd.DataFrame([input_data])

    # Example grouping of product families into 'FOODS' and 'NON-FOODS'
    food_families = ['BEVERAGES', 'FOODS']
    input_df['family'] = np.where(input_df['family'].isin(food_families), 'FOODS', input_df['family'])

    # Separate categorical and numerical columns
    input_df_cat = input_df[categorical_columns].copy()
    input_df_num = input_df.drop(columns=categorical_columns)

    # Impute missing values
    input_df_cat_imputed = cat_imputer.transform(input_df_cat)
    input_df_num_imputed = num_imputer.transform(input_df_num)

    # Encode categorical variables
    input_df_cat_encoded = encoder.transform(input_df_cat_imputed).toarray()
    encoded_feature_names = encoder.get_feature_names_out(categorical_columns)

    # Scale numerical variables
    input_df_num_scaled = scaler.transform(input_df_num_imputed)

    # Combine numeric and categorical features
    input_processed = np.hstack([input_df_num_scaled, input_df_cat_encoded])

    # Predict sales
    predictions = model.predict(input_processed)

    # Display result
    st.success(f"Predicted sales: {predictions[0]:.2f}")
```

Replace the grouping logic, imputers, encoder, and scaler with the exact preprocessing steps you used during training. This ensures consistency between training and inference.


## Part VI – Run the Streamlit App

After completing your `app.py` script, launch the application with:

```bash
streamlit run app.py
```

This command starts a local web server and opens the Streamlit interface in your default browser. Enter values in the fields and click **Predict** to see the estimated sales. Don’t forget to push your code to GitHub as part of the exercise submission.
